<a href="https://colab.research.google.com/github/Dasheeer/lab_REST-API/blob/main/REST_API_%D0%A1%D1%82%D1%83%D0%B4%D0%B5%D0%BD%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B9_%D0%B6%D1%83%D1%80%D0%BD%D0%B0%D0%BB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import time
import threading
import nest_asyncio
import uvicorn

from fastapi import FastAPI, HTTPException, Path, Query
from pydantic import BaseModel, Field, ConfigDict
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey
from sqlalchemy.orm import declarative_base, sessionmaker
from pyngrok import ngrok

# Разрешаем запуск сервера в Google Colab
nest_asyncio.apply()

# 1. Создаём приложение
app = FastAPI(
    title="API студенческого журнала",
    version="1.0.0"
)
# 2. Подключаем базу данных SQLite
DATABASE_URL = "sqlite:///students.db"

engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False}
)

SessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)

Base = declarative_base()


# 3. Модель студента
class Student(Base):
    __tablename__ = "students"

    id = Column(Integer, primary_key=True, index=True)
    name = Column(String, nullable=False)
    group_name = Column(String, nullable=False)


# 4. Модель оценки
class Grade(Base):
    __tablename__ = "grades"

    id = Column(Integer, primary_key=True, index=True)
    student_id = Column(
        Integer,
        ForeignKey("students.id"),
        nullable=False
    )
    subject = Column(String, nullable=False)
    grade = Column(Integer, nullable=False)


# Создаём таблицы, если их ещё нет
Base.metadata.create_all(bind=engine)


# 5. Схема данных студента
class StudentCreate(BaseModel):
    name: str = Field(
        min_length=1,
        description="Фамилия и имя студента",
        examples=["Анна Смирнова"]
    )
    group_name: str = Field(
        min_length=1,
        description="Название учебной группы",
        examples=["БСТ2556"]
    )

    model_config = ConfigDict(
        json_schema_extra={
            "example": {
                "name": "Анна Смирнова",
                "group_name": "БСТ2556"
            }
        }
    )


# 6. Схема данных оценки
class GradeCreate(BaseModel):
    subject: str = Field(
        min_length=1,
        description="Название учебной дисциплины",
        examples=["Базы данных"]
    )
    grade: int = Field(
        ge=2,
        le=5,
        description="Оценка по пятибалльной шкале: от 2 до 5",
        examples=[5]
    )

    model_config = ConfigDict(
        json_schema_extra={
            "example": {
                "subject": "Базы данных",
                "grade": 5
            }
        }
    )


# 7. Преобразование данных в JSON
def student_to_dict(student):
    return {
        "id": student.id,
        "name": student.name,
        "group_name": student.group_name
    }


def grade_to_dict(grade):
    return {
        "id": grade.id,
        "student_id": grade.student_id,
        "subject": grade.subject,
        "grade": grade.grade
    }


# 8. Получить список всех студентов — GET
@app.get(
    "/students",
    tags=["Студенты"],
    summary="Получить список студентов",
    description=(
        "Возвращает список студентов. "
        "Можно фильтровать по учебной группе "
        "и сортировать по имени."
    )
)
def get_students(
    group_name: str | None = Query(
        default=None,
        description="Фильтр по названию учебной группы",
        examples=["БСТ2556"]
    ),
    sort_order: str = Query(
        default="asc",
        pattern="^(asc|desc)$",
        description=(
            "Порядок сортировки по имени: "
            "asc — по алфавиту, desc — в обратном порядке"
        )
    )
):
    db = SessionLocal()
    try:
        query = db.query(Student)

        # Фильтрация по учебной группе
        if group_name:
            query = query.filter(
                Student.group_name == group_name
            )

        # Сортировка по имени
        if sort_order == "asc":
            query = query.order_by(Student.name.asc())
        else:
            query = query.order_by(Student.name.desc())

        students = query.all()

        return [student_to_dict(s) for s in students]
    finally:
        db.close()



# 9. Получить информацию о студенте — GET
@app.get(
    "/students/{student_id}",
    tags=["Студенты"],
    summary="Получить информацию о студенте",
    description="Возвращает данные студента по его идентификатору."
)
def get_student(
    student_id: int = Path(
        ...,
        description="Уникальный идентификатор студента"
    )
):
    db = SessionLocal()
    try:
        student = db.query(Student).filter(
            Student.id == student_id
        ).first()

        if student is None:
            raise HTTPException(
                status_code=404,
                detail="Студент не найден"
            )

        return student_to_dict(student)
    finally:
        db.close()


# 10. Добавить студента — POST
@app.post(
    "/students",
    tags=["Студенты"],
    status_code=201,
    summary="Добавить студента",
    description="Создаёт новую запись о студенте."
)
def create_student(data: StudentCreate):
    db = SessionLocal()
    try:
        student = Student(
            name=data.name,
            group_name=data.group_name
        )
        db.add(student)
        db.commit()
        db.refresh(student)

        return student_to_dict(student)
    finally:
        db.close()


# 11. Изменить данные студента — PUT
@app.put(
    "/students/{student_id}",
    tags=["Студенты"],
    summary="Изменить данные студента",
    description="Обновляет имя и учебную группу студента."
)
def update_student(student_id: int, data: StudentCreate):
    db = SessionLocal()
    try:
        student = db.query(Student).filter(
            Student.id == student_id
        ).first()

        if student is None:
            raise HTTPException(
                status_code=404,
                detail="Студент не найден"
            )

        student.name = data.name
        student.group_name = data.group_name

        db.commit()
        db.refresh(student)

        return student_to_dict(student)
    finally:
        db.close()


# 12. Удалить студента — DELETE
@app.delete(
    "/students/{student_id}",
    tags=["Студенты"],
    summary="Удалить студента",
    description="Удаляет студента и все его оценки."
)
def delete_student(student_id: int):
    db = SessionLocal()
    try:
        student = db.query(Student).filter(
            Student.id == student_id
        ).first()

        if student is None:
            raise HTTPException(
                status_code=404,
                detail="Студент не найден"
            )

        db.query(Grade).filter(
            Grade.student_id == student_id
        ).delete()

        db.delete(student)
        db.commit()

        return {"message": "Студент успешно удалён"}
    finally:
        db.close()


# 13. Добавить оценку студенту — POST
@app.post(
    "/students/{student_id}/grades",
    tags=["Студенты"],
    status_code=201,
    summary="Добавить оценку студенту",
    description="Добавляет студенту оценку по дисциплине."
)
def create_grade(student_id: int, data: GradeCreate):
    db = SessionLocal()
    try:
        student = db.query(Student).filter(
            Student.id == student_id
        ).first()

        if student is None:
            raise HTTPException(
                status_code=404,
                detail="Студент не найден"
            )

        grade = Grade(
            student_id=student_id,
            subject=data.subject,
            grade=data.grade
        )

        db.add(grade)
        db.commit()
        db.refresh(grade)

        return grade_to_dict(grade)
    finally:
        db.close()


# 14. Получить оценки студента — GET
@app.get(
    "/students/{student_id}/grades",
    tags=["Студенты"],
    summary="Получить оценки студента",
    description="Возвращает все оценки студента по дисциплинам."
)
def get_grades(student_id: int):
    db = SessionLocal()
    try:
        student = db.query(Student).filter(
            Student.id == student_id
        ).first()

        if student is None:
            raise HTTPException(
                status_code=404,
                detail="Студент не найден"
            )

        grades = db.query(Grade).filter(
            Grade.student_id == student_id
        ).all()

        return [grade_to_dict(g) for g in grades]
    finally:
        db.close()


# 15. Изменить оценку — PUT
@app.put(
    "/grades/{grade_id}",
    tags=["Студенты"],
    summary="Изменить оценку",
    description="Изменяет дисциплину или балл существующей оценки."
)
def update_grade(grade_id: int, data: GradeCreate):
    db = SessionLocal()
    try:
        grade = db.query(Grade).filter(
            Grade.id == grade_id
        ).first()

        if grade is None:
            raise HTTPException(
                status_code=404,
                detail="Оценка не найдена"
            )

        grade.subject = data.subject
        grade.grade = data.grade

        db.commit()
        db.refresh(grade)

        return grade_to_dict(grade)
    finally:
        db.close()


# 16. Удалить оценку — DELETE
@app.delete(
    "/grades/{grade_id}",
    tags=["Студенты"],
    summary="Удалить оценку",
    description="Удаляет выбранную оценку из журнала."
)
def delete_grade(grade_id: int):
    db = SessionLocal()
    try:
        grade = db.query(Grade).filter(
            Grade.id == grade_id
        ).first()

        if grade is None:
            raise HTTPException(
                status_code=404,
                detail="Оценка не найдена"
            )

        db.delete(grade)
        db.commit()

        return {"message": "Оценка успешно удалена"}
    finally:
        db.close()


# 17. Запуск сервера
server = uvicorn.Server(
    uvicorn.Config(
        app,
        host="0.0.0.0",
        port=8002,
        log_level="warning"
    )
)

api_thread = threading.Thread(
    target=server.run,
    daemon=True
)
api_thread.start()

# Проверяем, запустился ли сервер
for _ in range(30):
    if server.started:
        break
    time.sleep(0.5)

if not server.started:
    raise RuntimeError(
        "Сервер не запустился. Возможно, порт 8002 занят. "
        "Перезапусти среду выполнения Colab и попробуй снова."
    )

# 18. Создаём публичную ссылку для Swagger
ngrok.kill()
tunnel = ngrok.connect(8002)
public_url = tunnel.public_url

print("API успешно запущен!")
print("Ссылка на API:", public_url)
print("Документация Swagger:", public_url + "/docs")
print("База данных SQLite готова к работе.")


API успешно запущен!
Ссылка на API: https://c843-34-24-83-60.ngrok-free.app
Документация Swagger: https://c843-34-24-83-60.ngrok-free.app/docs
База данных SQLite готова к работе.
